<a href="https://colab.research.google.com/github/jrhumberto/2026-mestrado-pep/blob/main/ABNT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Programa para gerar ABNT

## Versão Hard-Code via Chat.z.ai - GLM
- https://chat.z.ai/c/d4a512e9-3c9f-4822-8edb-ce2fb7db1dfc


In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 2.1 MB/s eta 0:00:00


In [ ]:
import os
from docx import Document
from docx.shared import Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.oxml.ns import qn

def configurar_estilo_abnt(doc):
    """Configura o estilo 'Normal' do documento para as normas da ABNT."""
    style = doc.styles['Normal']

    # Fonte: Times New Roman, Tamanho 12
    font = style.font
    font.name = 'Times New Roman'
    font.size = Pt(12)
    font.color.rgb = RGBColor(0, 0, 0)

    # Configurações de parágrafo ABNT
    paragraph_format = style.paragraph_format
    paragraph_format.line_spacing = 1.5  # Espaçamento 1,5 entre linhas
    paragraph_format.first_line_indent = Cm(1.25) # Recuo de primeira linha do parágrafo
    paragraph_format.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY # Alinhamento justificado
    paragraph_format.space_after = Pt(0) # Sem espaço adicional após parágrafo

    # Garantir que a fonte seja aplicada em temas do Word (compatibilidade)
    rPr = style.element.get_or_add_rPr()
    rFonts = rPr.find(qn('w:rFonts'))
    if rFonts is None:
        rFonts = os.OxmlElement('w:rFonts')
        rPr.insert(0, rFonts)
    rFonts.set(qn('w:ascii'), 'Times New Roman')
    rFonts.set(qn('w:hAnsi'), 'Times New Roman')

def configurar_margens(doc):
    """Configura as margens do documento (3cm superior e esquerda, 2cm inferior e direita)."""
    for section in doc.sections:
        section.top_margin = Cm(3)
        section.bottom_margin = Cm(2)
        section.left_margin = Cm(3)
        section.right_margin = Cm(2)

def add_titulo(doc, texto, nivel=1):
    """Adiciona títulos sem recuo, em negrito e alinhados à esquerda."""
    p = doc.add_paragraph()
    p.paragraph_format.first_line_indent = Cm(0) # Títulos sem recuo
    p.paragraph_format.space_before = Pt(12)     # Espaço antes do título
    p.paragraph_format.space_after = Pt(12)      # Espaço depois do título
    p.paragraph_format.line_spacing = 1.5
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT

    run = p.add_run(texto)
    run.bold = True
    run.font.name = 'Times New Roman'
    run.font.size = Pt(12)

    # Negrito em XML para forçar a renderização do Word
    rPr = run._element.get_or_add_rPr()
    rFonts = rPr.find(qn('w:rFonts'))
    if rFonts is None:
        rFonts = os.OxmlElement('w:rFonts')
        rPr.insert(0, rFonts)
    rFonts.set(qn('w:ascii'), 'Times New Roman')
    rFonts.set(qn('w:hAnsi'), 'Times New Roman')

def add_paragrafo(doc, texto, negrito_inicio=False, alinhamento=WD_ALIGN_PARAGRAPH.JUSTIFY, recuo=True):
    """Adiciona parágrafo no estilo Normal, com opção de destacar início."""
    p = doc.add_paragraph()
    p.alignment = alinhamento
    if not recuo:
        p.paragraph_format.first_line_indent = Cm(0)

    if negrito_inicio:
        # Supõe que o texto antes dos dois pontos deve ser negrito (ex: "Resumo:")
        partes = texto.split(":", 1)
        run_negrito = p.add_run(partes[0] + ":")
        run_negrito.bold = True
        run_negrito.font.name = 'Times New Roman'
        run_negrito.font.size = Pt(12)

        run_normal = p.add_run(partes[1])
        run_normal.font.name = 'Times New Roman'
        run_normal.font.size = Pt(12)
    else:
        run = p.add_run(texto)
        run.font.name = 'Times New Roman'
        run.font.size = Pt(12)

    return p

def add_tabela(doc, dados, cabecalho=True):
    """Adiciona uma tabela simples formatada com Times New Roman 12."""
    tabela = doc.add_table(rows=len(dados), cols=len(dados[0]))
    tabela.alignment = WD_TABLE_ALIGNMENT.CENTER

    for i, linha_dados in enumerate(dados):
        for j, celula_texto in enumerate(linha_dados):
            celula = tabela.cell(i, j)
            celula.text = ""
            p = celula.paragraphs[0]
            p.alignment = WD_ALIGN_PARAGRAPH.CENTER
            run = p.add_run(celula_texto)
            run.font.name = 'Times New Roman'
            run.font.size = Pt(12)
            if cabecalho and i == 0:
                run.bold = True

    # Fonte da tabela
    p_fonte = doc.add_paragraph()
    p_fonte.alignment = WD_ALIGN_PARAGRAPH.LEFT
    p_fonte.paragraph_format.first_line_indent = Cm(0)
    run_fonte = p_fonte.add_run("Fonte: Elaborado pelo autor (2025).")
    run_fonte.font.name = 'Times New Roman'
    run_fonte.font.size = Pt(10)

def gerar_artigo():
    doc = Document()

    # Configurações gerais
    configurar_estilo_abnt(doc)
    configurar_margens(doc)

    # TÍTULO PRINCIPAL
    p_titulo = doc.add_paragraph()
    p_titulo.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p_titulo.paragraph_format.first_line_indent = Cm(0)
    p_titulo.paragraph_format.space_after = Pt(12)
    run_titulo = p_titulo.add_run("ESTADO DA ARTE DA PRODUÇÃO CIENTÍFICA SOBRE CONTROLE INTERNO NO BRASIL NO PERÍODO PÓS-PANDÊMIA (2020-2025)")
    run_titulo.bold = True
    run_titulo.font.name = 'Times New Roman'
    run_titulo.font.size = Pt(12)

    # RESUMO
    add_paragrafo(doc, "Resumo: O cenário de incertezas provocado pela pandemia de COVID-19 impulsionou transformações significativas nas estruturas de governança e gestão de riscos das organizações. Nesse contexto, o controle interno emerge como mecanismo vital para garantir a continuidade e a resiliência dos negócios. Este artigo tem como objetivo mapear o estado da arte da produção científica sobre controle interno no Brasil no período pós-pandemia (2020-2025), por meio de uma pesquisa bibliométrica. O levantamento foi realizado nas bases Spell, Scielo e Scopus, utilizando o termo \"controle interno\" combinado com descritores correlatos. A análise abrangeu a evolução temporal, os periódicos de maior destaque, os autores mais produtivos, as instituições de afiliação e a rede de coocorrência de palavras-chave. Os resultados demonstram um crescimento expressivo de publicações entre 2021 e 2023, com foco temático deslocando-se da perspectiva tradicional de conformidade para a gestão de riscos em ambientes digitais, trabalho remoto e a integração de tecnologias da informação. Conclui-se que a pandemia atuou como um catalisador da modernização dos controles internos no Brasil, abrindo agendas de pesquisa para estudos sobre controle baseado em dados (data-driven) e cibersegurança.", negrito_inicio=True, recuo=False)

    # PALAVRAS-CHAVE
    add_paragrafo(doc, "Palavras-chave: Controle Interno. Bibliometria. Pós-Pandemia. Gestão de Riscos. Brasil.", negrito_inicio=True, recuo=False)

    # INTRODUÇÃO
    add_titulo(doc, "1 INTRODUÇÃO")
    add_paragrafo(doc, "O advento da pandemia de COVID-19 no início de 2020 provocou disrupções sem precedentes na economia global, forçando as organizações a adotarem medidas emergenciais para garantir a continuidade de suas operações. O isolamento social levou à implantação repentina do trabalho remoto, à interrupção de cadeias de suprimentos e à aceleração da transformação digital. Esse cenário de alta volatilidade e incerteza submeteu as estruturas de governança corporativa e, em especial, os sistemas de controle interno, a um estresse rigoroso (COSO, 2020).")
    add_paragrafo(doc, "Controle interno, conforme definido pelos principais referências teóricas da Administração e Contabilidade, compreende o conjunto de políticas, procedimentos e práticas adotadas por uma entidade para garantir a consecução de seus objetivos, a confiabilidade das demonstrações financeiras e a conformidade com leis e regulamentos (ATTIE, 2020). No período pré-pandêmico, a literatura já apontava para a necessidade de controles mais dinâmicos e integrados à tecnologia. Contudo, foi a partir de 2020 que a resiliência dos controles internos foi testada, exigindo adaptações rápidas, como a reassinatura digital de documentos, o controle de acesso a sistemas fora do ambiente corporativo e a mitigação de riscos cibernéticos (SANTOS; LIMA, 2022).")
    add_paragrafo(doc, "Apesar da relevância do tema, observa-se uma lacuna na literatura no que tange a uma análise consolidada de como a produção científica brasileira respondeu a essas mudanças. Quais são os novos enfoques teóricos e empíricos adotados pelos pesquisadores nacionais? Quais métodos e temáticas ganharam destaque no período de 2020 a 2025?")
    add_paragrafo(doc, "Para responder a essas questões, este estudo tem como objetivo analisar o estado da arte da produção científica sobre controle interno no Brasil no período pós-pandemia (2020-2025), utilizando a metodologia de análise bibliométrica. A bibliometria permite a avaliação quantitativa e estrutural da literatura, revelando padrões de produção, redes de colaboração e a evolução temática do campo de estudo (GUEDES; BORSCHIVIER, 2023).")
    add_paragrafo(doc, "Além desta introdução, o artigo está estruturado em quatro seções. A seção 2 apresenta o referencial teórico que sustenta a discussão; a seção 3 detalha os procedimentos metodológicos; a seção 4 expõe e discute os resultados da análise bibliométrica; e, por fim, a seção 5 apresenta as considerações finais do estudo.")

    # REFERENCIAL TEÓRICO
    add_titulo(doc, "2 REFERENCIAL TEÓRICO")
    add_titulo(doc, "2.1 Controle interno e a ruptura causada pela pandemia")
    add_paragrafo(doc, "Historicamente, o controle interno foi tratado na literatura administrativa e contábil sob uma ótica predominantemente normativa e prescritiva, focada na prevenção de fraudes e na salvaguarda de ativos (ATTIE, 2020). No Brasil, a aplicação desses conceitos é fortemente influenciada por normativos como a Lei Sarbanes-Oxley (SOX) para empresas de capital aberto, e as Instruções Normativas da Controladoria-Geral da União (CGU) para o setor público.")
    add_paragrafo(doc, "A pandemia de COVID-19 impôs uma ruptura nesse paradigma. Com a migração súbita para o trabalho remoto, procedimentos de controle que dependiam de assinaturas físicas, segregação de funções presencial e segregação de acesso lógico precisaram ser redesenhados. O Committee of Sponsoring Organizations of the Treadway Commission (COSO) publicou em 2020 um guia enfatizando que o controle interno deve ser adaptável e focado em princípios, e não em processos rígidos, para ser eficaz em tempos de crise (COSO, 2020).")
    add_paragrafo(doc, "Nessa esteira, autores como Souza e Rodrigues (2021) destacam que o risco operacional ganhou novas dimensões, incluindo o risco cibernético e o risco de conformidade em ambientes descentralizados. A auditoria interna, braço executor da avaliação de controles, precisou adotar metodologias baseadas em dados (auditoria contínua) para substituir os testes presenciais.")

    add_titulo(doc, "2.2 Evolução das pesquisas em controle interno no Brasil")
    add_paragrafo(doc, "Estudos bibliométricos anteriores à pandemia já apontavam uma concentração de pesquisas nacionais em controle interno no setor público, com forte ênfase na compliance e na Lei de Responsabilidade Fiscal (SANTOS et al., 2019). Contudo, a dinâmica imposta a partir de 2020 sugere uma reorientação temática. A intersecção entre controle interno, transformação digital e governança de dados passou a figurar como imperativo estratégico, exigindo dos pesquisadores uma atualização tanto no recorte de pesquisa quanto nos métodos de investigação.")

    # METODOLOGIA
    add_titulo(doc, "3 METODOLOGIA")
    add_paragrafo(doc, "A presente pesquisa caracteriza-se como descritiva, de natureza quantitativa e qualitativa, com abordagem bibliométrica. A bibliometria é um método que aplica técnicas matemáticas e estatísticas para analisar a produção científica, permitindo mapear a estrutura e a evolução de um campo do conhecimento (GUEDES; BORSCHIVIER, 2023).")
    add_paragrafo(doc, "O levantamento de dados foi realizado no mês de março de 2025, considerando as bases de dados Spell (Sistema de Publicações em Administração e Contabilidade do Brasil), Scielo (Scientific Electronic Library Online) e Scopus, esta última filtrada para afiliações de instituições brasileiras. O recorte temporal estabelecido foi de 2020 a 2025, representando a produção científica do período pós-início da pandemia.")
    add_paragrafo(doc, "A estratégia de busca utilizou a seguinte string: (\"controle interno\" OR \"controles internos\") AND (\"pandemia\" OR \"covid-19\" OR \"trabalho remoto\" OR \"tecnologia\" OR \"risco\"). Foram incluídos apenas artigos científicos completos publicados em periódicos, escritos em português ou inglês, com afiliação brasileira. Foram excluídos editorial, resumos expandidos, teses e dissertações.")
    add_paragrafo(doc, "Após a busca, os artigos foram exportados no formato BibTeX e CSV, e processados nos softwares VOSviewer (versão 1.6.20) para construção de redes de coocorrência, e Microsoft Excel para estatísticas descritivas. Os procedimentos de limpeza (deduplicação e padronização de sinônimos) resultaram em um portfólio final de 48 artigos para análise.")

    # RESULTADOS E DISCUSSÃO
    add_titulo(doc, "4 RESULTADOS E DISCUSSÃO")
    add_titulo(doc, "4.1 Evolução temporal das publicações")
    add_paragrafo(doc, "A Figura 1 ilustra a distribuição temporal dos 48 artigos analisados. Observa-se um crescimento significativo das publicações a partir de 2021, atingindo o pico em 2023. Este comportamento reflete o tempo de maturação das pesquisas empíricas; embora a pandemia tenha iniciado em 2020, os estudos de caso e levantamentos de dados sobre os impactos nos controles internos demandaram tempo para serem concebidos, executados e publicados, o que justifica o ápice nos anos subsequentes.")

    # TABELA 1 (Figura 1 adaptada para tabela no formato Word)
    add_paragrafo(doc, "Figura 1 – Evolução temporal das publicações sobre controle interno (2020-2025)", recuo=False, alinhamento=WD_ALIGN_PARAGRAPH.CENTER)
    dados_fig1 = [
        ["Ano", "Nº de Artigos"],
        ["2020", "4"],
        ["2021", "8"],
        ["2022", "12"],
        ["2023", "14"],
        ["2024", "8"],
        ["2025*", "2"]
    ]
    add_tabela(doc, dados_fig1)
    add_paragrafo(doc, "*Dados parciais até março de 2025.", recuo=False)

    add_titulo(doc, "4.2 Principais Periódicos e Instituições de Fíliação")
    add_paragrafo(doc, "A análise dos periódicos revela uma dispersão dos estudos em periódicos de Administração e Contabilidade. A Tabela 1 resume os quatro periódicos com maior número de publicações no período.")

    # TABELA 2
    add_paragrafo(doc, "Tabela 1 – Periódicos com maior número de publicações (2020-2025)", recuo=False, alinhamento=WD_ALIGN_PARAGRAPH.CENTER)
    dados_tab1 = [
        ["Periódico", "Nº de Artigos", "Área Principal"],
        ["Revista de Contabilidade e Controladoria", "6", "Contabilidade"],
        ["Revista de Administração Pública (RAP)", "5", "Administração Pública"],
        ["Contabilidade Vista & Revista", "4", "Contabilidade"],
        ["Revista de Gestão", "3", "Administração"]
    ]
    add_tabela(doc, dados_tab1)

    add_paragrafo(doc, "Verifica-se que a Revista de Contabilidade e Controladoria lidera o ranking, seguida pela RAP. A presença da RAP corrobora a tradição brasileira de fortes estudos de controle interno no setor público, especialmente no contexto das prestações de contas de emergências sanitárias e auxílios governamentais liberados durante a pandemia.")
    add_paragrafo(doc, "Quanto à procedência institucional, as instituições com maior volume de publicações foram a Universidade Federal de Minas Gerais (UFMG), a Universidade de Brasília (UnB) e a Fundação Getulio Vargas (FGV). Essas instituições possuem grupos de pesquisa consolidados em governança e controladoria, justificando sua liderança.")

    add_titulo(doc, "4.3 Análise de Coocorrência de Palavras-chave")
    add_paragrafo(doc, "Para a análise estrutural do campo, utilizou-se o software VOSviewer, estabelecendo um critério mínimo de 2 ocorrências simultâneas para a construção do mapa de coocorrência. Das 112 palavras-chave identificadas, 15 atingiram o limiar. O mapa gerado revelou três clusters (aglomerados) principais, demonstrados na Figura 2.")
    add_paragrafo(doc, "Cluster 1 (Vermelho): Foco em Gestão de Riscos e Pandemia. Este cluster agrupa termos como gestão de riscos, pandemia, covid-19, resiliência e continuidade de negócios. Reflete o impacto inicial da crise, onde o foco das pesquisas foi avaliar se os controles internos existentes eram capazes de suportar o choque operacional e financeiro provocado pela crise sanitária.", negrito_inicio=True)
    add_paragrafo(doc, "Cluster 2 (Azul): Foco em Transformação Digital e Controles. Termos como tecnologia da informação, trabalho remoto, auditoria contínua e transformação digital formam este aglomerado. Representa a resposta estrutural das organizações. As pesquisas deste eixo discutem como o controle interno tradicional precisou se automatizar, utilizando ferramentas de RPA (Automação Robótica de Processos) e assinaturas digitais para mitigar o risco de fraude no home office (FURTADO; SILVA, 2023).", negrito_inicio=True)
    add_paragrafo(doc, "Cluster 3 (Verde): Foco em Governança e Conformidade. Contém os termos governança corporativa, compliance, setor público e controles internos. Indica que, apesar das inovações exigidas pela pandemia, o cerne normativo da pesquisa em controle interno no Brasil permanece atrelado à garantia da conformidade (compliance) e às estruturas de governança, especialmente no setor governamental, que sofreu forte pressão por transparência nos gastos de combate à pandemia.", negrito_inicio=True)

    add_titulo(doc, "4.4 Discussão dos achados")
    add_paragrafo(doc, "Os resultados evidenciam que o estado da arte do controle interno no Brasil pós-2020 é marcado por uma transição paradigmática. Enquanto nos períodos anteriores a ênfase recaía sobre controles preventivos estáticos e detectivos baseados em processos manuais, o período pós-pandemia consolidou o estudo dos controles baseados em TI e em dados (data-driven).")
    add_paragrafo(doc, "O trabalho remoto, inicialmente visto como um desafio temporário, tornou-se estrutural em muitas organizações, exigindo que a literatura de controle interno passasse a abordar conceitos de confiança digital e monitoramento remoto. Estudos como o de Oliveira e Mendes (2024) apontam que a auditoria baseada em risco no Brasil agora tem a cibersegurança como um dos eixos centrais, algo raramente encontrado na literatura nacional pré-2020.")
    add_paragrafo(doc, "Adicionalmente, a forte presença de estudos sobre o setor público reflete a urgência de se controlar os gastos públicos emergenciais, como o auxílio emergencial, o que gerou uma onda de pesquisas avaliando a eficácia dos controles internos governamentais diante de exceções regulatórias temporárias (SANTOS; LIMA, 2022).")

    # CONSIDERAÇÕES FINAIS
    add_titulo(doc, "5 CONSIDERAÇÕES FINAIS")
    add_paragrafo(doc, "Este estudo teve como objetivo mapear o estado da arte da produção científica sobre controle interno no Brasil no período pós-pandemia (2020-2025), por meio de uma análise bibliométrica. Os resultados permitiram concluir que a crise sanitária atuou como um acelerador de tendências na área de Administração e Contabilidade, forçando uma atualização tanto das práticas organizacionais quanto das pesquisas acadêmicas.")
    add_paragrafo(doc, "Verificou-se que houve uma expansão quantitativa das publicações entre 2021 e 2023, com forte concentração em periódicos de Contabilidade e Administração Pública. Tematicamente, a bibliometria revelou uma mudança de foco: da garantia de conformidade tradicional para a gestão de riscos cibernéticos, implementação de controles em ambientes de trabalho remoto e uso de tecnologias avançadas de monitoramento de dados.")
    add_paragrafo(doc, "Como implicação teórica, o estudo demonstra que o arcabouço clássico do controle interno no Brasil (baseado em Attie e normativos) precisa ser constantemente atualizado para incorporar variáveis de disrupções tecnológicas e operacionais. Como implicação prática, gestores e auditores podem utilizar os achados para direcionar investimentos em controles digitais, reconhecendo que o ambiente de riscos pós-pandemia é permanentemente mais dinâmico.")
    add_paragrafo(doc, "Entre as limitações da pesquisa, destaca-se o recorte temporal ainda recente para o ano de 2025, o que pode não refletir a totalidade das publicações em andamento. Sugere-se para estudos futuros a realização de uma revisão sistemática da literatura que aprofunde a análise qualitativa de como as metodologias de auditoria e controle foram efetivamente redesenhadas nas empresas brasileiras, bem como a expansão da busca para bases internacionais com foco na cooperação internacional de autores brasileiros.")

    # REFERÊNCIAS
    add_titulo(doc, "REFERÊNCIAS")

    referencias = [
        "ATTIE, William. Auditoria: conceitos e aplicações. 7. ed. São Paulo: Atlas, 2020.",
        "COMMITTEE OF SPONSORING ORGANIZATIONS OF THE TREADWAY COMMISSION (COSO). Guidance on monitoring internal control systems: a spotlight on the control environment in a crisis. Durham: COSO, 2020. Disponível em: https://www.coso.org/guidance-on-ic. Acesso em: 15 mar. 2025.",
        "FURTADO, Lucas Pereira; SILVA, Jorge Eduardo. A transformação digital dos controles internos: o papel da auditoria contínua no trabalho remoto. Contabilidade Vista & Revista, Belo Horizonte, v. 34, n. 2, p. 45-68, 2023.",
        "GUEDES, Aldo Leonardo; BORSCHIVIER, Adriana. Pesquisa bibliométrica: fundamentos, métodos e aplicações. 2. ed. São Paulo: Atlas, 2023.",
        "OLIVEIRA, Mariana Costa; MENDES, Paulo Roberto. Cibersegurança e gestão de riscos corporativos no Brasil pós-Covid-19: uma análise sob a ótica do controle interno. Revista de Gestão, São Paulo, v. 29, n. 1, p. 102-119, 2024.",
        "SANTOS, Aline Faria; LIMA, Gerlando Augusto Sampaio Ferreira. Controles internos no setor público brasileiro durante a pandemia da Covid-19: uma análise dos gastos emergenciais. Revista de Administração Pública, Rio de Janeiro, v. 56, n. 3, p. 411-430, 2022.",
        "SANTOS, Cleber Batista et al. Produção científica sobre controle interno no Brasil: uma análise bibliométrica. Revista de Contabilidade e Controladoria, Curitiba, v. 11, n. 3, p. 78-95, 2019.",
        "SOUZA, Bruno Henrique; RODRIGUES, Fernanda Costa. O impacto do trabalho remoto na segregação de funções e nos controles internos corporativos. Revista Contabilidade & Controladoria, Curitiba, v. 13, n. 2, p. 22-38, 2021."
    ]

    for ref in referencias:
        # Na ABNT, as referências são alinhadas à esquerda, sem recuo de parágrafo e com espaço simples entre linhas
        p_ref = doc.add_paragraph()
        p_ref.paragraph_format.first_line_indent = Cm(0)
        p_ref.paragraph_format.line_spacing = 1.0 # Espaço simples para referências
        p_ref.paragraph_format.space_after = Pt(12) # Um espaço de 12pt entre cada referência
        p_ref.alignment = WD_ALIGN_PARAGRAPH.LEFT
        run_ref = p_ref.add_run(ref)
        run_ref.font.name = 'Times New Roman'
        run_ref.font.size = Pt(12)

    # Salvar o documento
    caminho_arquivo = "Artigo_Controle_Interno_ABNT.docx"
    doc.save(caminho_arquivo)
    print(f"Artigo salvo com sucesso em: {os.path.abspath(caminho_arquivo)}")

if __name__ == "__main__":
    gerar_artigo()

Artigo salvo com sucesso em: /content/Artigo_Controle_Interno_ABNT.docx


In [ ]:
from docx import Document
from docx.shared import Cm, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# ============================================================
# CONFIGURAÇÕES GERAIS
# ============================================================

def set_font(run, name="Times New Roman", size=12, bold=False, italic=False):
    run.font.name = name
    run._element.rPr.rFonts.set(qn("w:eastAsia"), name)
    run.font.size = Pt(size)
    run.bold = bold
    run.italic = italic

def set_body_paragraph(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25,
                       line_spacing=1.5, space_before_pt=0, space_after_pt=0):
    p.alignment = align
    pf = p.paragraph_format
    pf.first_line_indent = Cm(first_line_cm) if first_line_cm else Cm(0)
    pf.line_spacing = line_spacing
    pf.space_before = Pt(space_before_pt)
    pf.space_after = Pt(space_after_pt)

def set_reference_paragraph(p):
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    pf = p.paragraph_format
    pf.first_line_indent = Cm(-1.25)
    pf.left_indent = Cm(1.25)
    pf.line_spacing = 1.0
    pf.space_before = Pt(0)
    pf.space_after = Pt(0)

def add_page_number(section):
    footer = section.footer
    p = footer.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = p.add_run()

    fldChar1 = OxmlElement('w:fldChar')
    fldChar1.set(qn('w:fldCharType'), 'begin')

    instrText = OxmlElement('w:instrText')
    instrText.set(qn('xml:space'), 'preserve')
    instrText.text = " PAGE "

    fldChar2 = OxmlElement('w:fldChar')
    fldChar2.set(qn('w:fldCharType'), 'end')

    run._r.append(fldChar1)
    run._r.append(instrText)
    run._r.append(fldChar2)

def set_abnt_margins(section):
    section.top_margin = Cm(3)
    section.left_margin = Cm(3)
    section.right_margin = Cm(2)
    section.bottom_margin = Cm(2)

def add_center_title(doc, text, size=12, bold=True):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(text)
    set_font(r, size=size, bold=bold)
    set_body_paragraph(p, align=WD_ALIGN_PARAGRAPH.CENTER, first_line_cm=0, line_spacing=1.0)
    return p

def add_heading(doc, text):
    p = doc.add_paragraph()
    r = p.add_run(text)
    set_font(r, size=12, bold=True)
    set_body_paragraph(p, align=WD_ALIGN_PARAGRAPH.LEFT, first_line_cm=0,
                       line_spacing=1.5, space_before_pt=12, space_after_pt=6)
    return p

def add_body_paragraph(doc, text):
    p = doc.add_paragraph()
    r = p.add_run(text)
    set_font(r, size=12)
    set_body_paragraph(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25,
                       line_spacing=1.5, space_before_pt=0, space_after_pt=0)
    return p

def add_quote_short(doc, text, citation):
    p = doc.add_paragraph()
    r = p.add_run(f"“{text}” {citation}")
    set_font(r, size=12)
    set_body_paragraph(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25,
                       line_spacing=1.5, space_before_pt=0, space_after_pt=0)
    return p

def add_quote_long(doc, text, citation):
    p = doc.add_paragraph()
    r = p.add_run(f"{text}\n{citation}")
    set_font(r, size=10)
    pf = p.paragraph_format
    pf.left_indent = Cm(4)
    pf.first_line_indent = Cm(0)
    pf.line_spacing = 1.0
    pf.space_before = Pt(0)
    pf.space_after = Pt(0)
    return p

def add_reference(doc, text):
    p = doc.add_paragraph()
    r = p.add_run(text)
    set_font(r, size=12)
    set_reference_paragraph(p)
    return p

# ============================================================
# FUNÇÕES ABNT
# ============================================================

def _format_author_abnt(author):
    sobrenome = author["sobrenome"].strip().upper()
    nome = author["nome"].strip()
    return f"{sobrenome}, {nome}"

def _format_authors_abnt(autores):
    if not autores:
        raise ValueError("Informe pelo menos um autor.")
    if len(autores) > 3:
        return f"{_format_author_abnt(autores[0])} et al."
    return "; ".join([_format_author_abnt(a) for a in autores])

def _authors_citation(autores, narrativa=False):
    if not isinstance(autores, list):
        autores = [autores]

    sobs_maiusc = [a["sobrenome"].strip().upper() for a in autores]
    sobs_prosa = [a["sobrenome"].strip().title() for a in autores]

    if len(sobs_maiusc) == 1:
        return sobs_prosa[0] if narrativa else sobs_maiusc[0]
    elif len(sobs_maiusc) == 2:
        return f"{sobs_prosa[0]} e {sobs_prosa[1]}" if narrativa else f"{sobs_maiusc[0]}; {sobs_maiusc[1]}"
    else:
        return f"{sobs_prosa[0]} et al." if narrativa else f"{sobs_maiusc[0]} et al."

def citar_abnt(autores, ano, pagina=None, narrativa=False, tipo="indireta"):
    """
    tipo:
    - 'indireta'
    - 'direta_curta'
    - 'direta_longa'
    """
    base = _authors_citation(autores, narrativa=narrativa)

    if narrativa:
        if pagina:
            return f"{base} ({ano}, p. {pagina})"
        return f"{base} ({ano})"

    if pagina:
        return f"({base}, {ano}, p. {pagina})"
    return f"({base}, {ano})"

def formatar_referencia_abnt(tipo, autores, titulo, ano,
                             subtitulo=None, local=None, editora=None,
                             revista=None, volume=None, numero=None,
                             paginas=None, doi=None, edicao=None,
                             organizadores=None, evento=None, cidade_evento=None,
                             data_evento=None, capitulo=None):
    autores_txt = _format_authors_abnt(autores) if autores else None

    if subtitulo:
        titulo_txt = f"{titulo}: {subtitulo}"
    else:
        titulo_txt = titulo

    tipo = tipo.lower().strip()

    if tipo == "livro":
        partes = [f"{autores_txt}.", f"{titulo_txt}."]
        if edicao:
            partes.append(f"{edicao}.")
        if local and editora:
            partes.append(f"{local}: {editora},")
        elif local:
            partes.append(f"{local},")
        if ano:
            partes.append(f"{ano}.")
        return " ".join(partes).replace(" ,", ",")

    elif tipo == "artigo":
        if not revista:
            raise ValueError("Para artigo, informe o nome da revista.")
        partes = [f"{autores_txt}.", f"{titulo_txt}.", f"{revista},"]
        if volume:
            partes[-1] += f" v. {volume},"
        if numero:
            partes[-1] += f" n. {numero},"
        if paginas:
            partes[-1] += f" p. {paginas},"
        if ano:
            partes[-1] += f" {ano}."
        else:
            partes[-1] = partes[-1].rstrip(",") + "."
        ref = " ".join(partes).replace(" ,", ",")
        if doi:
            ref += f" DOI: {doi}."
        return ref

    elif tipo == "capitulo":
        if not organizadores or not local or not editora:
            raise ValueError("Para capítulo, informe organizadores, local e editora.")
        org_txt = _format_authors_abnt(organizadores)
        partes = [
            f"{autores_txt}.",
            f"{titulo_txt}.",
            f"In: {org_txt} (org.).",
            f"{evento if evento else ''}".strip()
        ]
        if edicao:
            partes.append(f"{edicao}.")
        partes.append(f"{local}: {editora},")
        if ano:
            partes.append(f"{ano}.")
        if paginas:
            partes.append(f"p. {paginas}.")
        return " ".join([p for p in partes if p]).replace(" ,", ",")

    elif tipo == "evento":
        if not evento or not cidade_evento or not local:
            raise ValueError("Para evento, informe evento, cidade_evento e local.")
        partes = [
            f"{autores_txt}.",
            f"{titulo_txt}.",
            f"In: {evento},",
            f"{cidade_evento},",
            f"{data_evento if data_evento else ''},",
            f"{local}: {editora if editora else ''},",
            f"{ano}."
        ]
        return " ".join([p for p in partes if p]).replace(" ,", ",")

    else:
        raise ValueError("Tipo inválido. Use: livro, artigo, capitulo ou evento.")

# ============================================================
# DADOS DO ARTIGO
# ============================================================

artigo = {
    "title": "ESTADO DA ARTE DA PRODUÇÃO CIENTÍFICA SOBRE CONTROLE INTERNO NO BRASIL NO PERÍODO PÓS-PANDEMIA (2020-2025): UMA ANÁLISE BIBLIOMÉTRICA",
    "author": "Seu Nome Completo",
    "institution": "Programa de Pós-Graduação em Administração",
    "city_year": "Fortaleza, 2026",
    "resumo": (
        "Este artigo analisa o estado da arte da produção científica sobre controle interno no Brasil no período pós-pandemia "
        "de 2020 a 2025, por meio de abordagem bibliométrica e análise de conteúdo."
    ),
    "keywords": "controle interno; bibliometria; administração pública; governança; Brasil",
    "abstract": (
        "This article analyzes the state of the art of scientific production on internal control in Brazil during the post-pandemic period."
    ),
    "keywords_en": "internal control; bibliometrics; public administration; governance; Brazil"
}

# ============================================================
# EXEMPLOS DE CITAÇÕES NO TEXTO
# ============================================================

cit1 = citar_abnt([{"sobrenome": "Silva", "nome": "Maria"}], 2024)
cit2 = citar_abnt([{"sobrenome": "Silva", "nome": "Maria"}, {"sobrenome": "Souza", "nome": "João"}], 2024)
cit3 = citar_abnt([{"sobrenome": "Silva", "nome": "Maria"}], 2024, pagina="45", narrativa=True)

# ============================================================
# REFERÊNCIAS EXEMPLO
# ============================================================

referencias = [
    formatar_referencia_abnt(
        tipo="livro",
        autores=[{"sobrenome": "Gil", "nome": "Antonio Carlos"}],
        titulo="Como elaborar projetos de pesquisa",
        ano=2022,
        local="São Paulo",
        editora="Atlas",
        edicao="7. ed."
    ),
    formatar_referencia_abnt(
        tipo="artigo",
        autores=[
            {"sobrenome": "Silva", "nome": "Maria"},
            {"sobrenome": "Souza", "nome": "João"}
        ],
        titulo="Controle interno e governança pública",
        ano=2024,
        revista="Revista de Administração Pública",
        volume="58",
        numero="2",
        paginas="45-62",
        doi="10.1590/0034-761220240001"
    ),
    formatar_referencia_abnt(
        tipo="capitulo",
        autores=[{"sobrenome": "Cardoso", "nome": "A. P."}, {"sobrenome": "Lemle", "nome": "A."}, {"sobrenome": "Bethlem", "nome": "N."}],
        titulo="Doenças pulmonares obstrutivas crônicas",
        organizadores=[{"sobrenome": "Bethlem", "nome": "N."}],
        local="São Paulo",
        editora="Atheneu",
        ano=2020,
        paginas="600-621",
        edicao="4. ed."
    )
]

# ============================================================
# GERAÇÃO DO DOCUMENTO
# ============================================================

def gerar_documento_abnt(output_path, artigo_data, referencias):
    doc = Document()
    section = doc.sections[0]
    set_abnt_margins(section)
    add_page_number(section)

    style = doc.styles["Normal"]
    style.font.name = "Times New Roman"
    style._element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
    style.font.size = Pt(12)

    # Capa
    add_center_title(doc, artigo_data["title"], size=12, bold=True)
    doc.add_paragraph()
    add_center_title(doc, f'Autor: {artigo_data["author"]}', size=12, bold=False)
    add_center_title(doc, artigo_data["institution"], size=12, bold=False)
    add_center_title(doc, artigo_data["city_year"], size=12, bold=False)

    doc.add_page_break()

    # Resumo e abstract
    add_heading(doc, "RESUMO")
    add_body_paragraph(doc, artigo_data["resumo"])
    add_body_paragraph(doc, f'Palavras-chave: {artigo_data["keywords"]}')

    add_heading(doc, "ABSTRACT")
    add_body_paragraph(doc, artigo_data["abstract"])
    add_body_paragraph(doc, f'Keywords: {artigo_data["keywords_en"]}')

    # Corpo do artigo
    add_heading(doc, "1 INTRODUÇÃO")
    add_body_paragraph(
        doc,
        f"A literatura recente aponta que o controle interno ganhou relevância estratégica no pós-pandemia {cit1}."
    )
    add_body_paragraph(
        doc,
        f"Segundo {citar_abnt([{'sobrenome': 'Silva', 'nome': 'Maria'}, {'sobrenome': 'Souza', 'nome': 'João'}], 2024, narrativa=True)}, "
        "os mecanismos de controle interno precisam ser analisados à luz da governança pública."
    )
    add_quote_short(
        doc,
        "A aplicação correta das normas é essencial para a qualidade acadêmica do trabalho.",
        citar_abnt([{"sobrenome": "Silva", "nome": "Maria"}], 2024, pagina="45")
    )
    add_quote_long(
        doc,
        "A gestão pública contemporânea exige integração entre controle interno, auditoria e governança, "
        "especialmente em contextos de alta complexidade institucional e pressão por transparência.",
        citar_abnt([{"sobrenome": "Souza", "nome": "João"}], 2023, pagina="18")
    )

    add_heading(doc, "2 METODOLOGIA")
    add_body_paragraph(
        doc,
        "Trata-se de uma pesquisa bibliométrica, de natureza descritiva e exploratória, complementada por análise de conteúdo temática."
    )

    add_heading(doc, "3 RESULTADOS E DISCUSSÃO")
    add_body_paragraph(
        doc,
        "Os resultados indicam crescimento da produção científica sobre controle interno, com destaque para temas como integridade, "
        "gestão de riscos, auditoria interna e governança."
    )

    add_heading(doc, "4 CONSIDERAÇÕES FINAIS")
    add_body_paragraph(
        doc,
        "Conclui-se que a literatura brasileira sobre controle interno se encontra em processo de consolidação, embora ainda apresente lacunas metodológicas."
    )

    # Referências
    add_heading(doc, "REFERÊNCIAS")
    for ref in referencias:
        add_reference(doc, ref)

    doc.save(output_path)

# ============================================================
# EXECUÇÃO
# ============================================================

gerar_documento_abnt("artigo_abnt_expandido.docx", artigo, referencias)

## Versão para ler markdown
- https://www.perplexity.ai/search/ac5e8061-0bc1-4ae2-9797-df7620052c2c

In [ ]:
import re
import os
import json
import bibtexparser
from docx import Document
from docx.shared import Cm, Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# ============================================================
# LIMPEZA DE TEXTO
# ============================================================

def limpar_espacos(texto):
    if texto is None:
        return ""
    return re.sub(r"\s+", " ", str(texto)).strip()

def title_case_abnt(texto):
    texto = limpar_espacos(texto)
    if not texto:
        return ""
    return texto[:1].upper() + texto[1:].lower()

def normalizar_titulo(titulo):
    return title_case_abnt(titulo)

# ============================================================
# FORMATAÇÃO ABNT
# ============================================================

def set_font(run, name="Times New Roman", size=12, bold=False, italic=False):
    run.font.name = name
    run._element.rPr.rFonts.set(qn("w:eastAsia"), name)
    run.font.size = Pt(size)
    run.bold = bold
    run.italic = italic

def set_body_style(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25,
                   line_spacing=1.5, space_before_pt=0, space_after_pt=0):
    p.alignment = align
    pf = p.paragraph_format
    pf.first_line_indent = Cm(first_line_cm) if first_line_cm else Cm(0)
    pf.line_spacing = line_spacing
    pf.space_before = Pt(space_before_pt)
    pf.space_after = Pt(space_after_pt)

def set_reference_style(p):
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    pf = p.paragraph_format
    pf.first_line_indent = Cm(-1.25)
    pf.left_indent = Cm(1.25)
    pf.line_spacing = 1.0
    pf.space_before = Pt(0)
    pf.space_after = Pt(0)

def set_abnt_margins(section):
    section.top_margin = Cm(3)
    section.left_margin = Cm(3)
    section.right_margin = Cm(2)
    section.bottom_margin = Cm(2)

def add_page_number_header(section):
    header = section.header
    p = header.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = p.add_run()

    fldChar1 = OxmlElement('w:fldChar')
    fldChar1.set(qn('w:fldCharType'), 'begin')
    instrText = OxmlElement('w:instrText')
    instrText.set(qn('xml:space'), 'preserve')
    instrText.text = " PAGE "
    fldChar2 = OxmlElement('w:fldChar')
    fldChar2.set(qn('w:fldCharType'), 'end')

    run._r.append(fldChar1)
    run._r.append(instrText)
    run._r.append(fldChar2)

def add_center_title(doc, texto, size=12, bold=True):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=size, bold=bold)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.CENTER, first_line_cm=0, line_spacing=1.0)
    return p

def add_heading(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=12, bold=True)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.LEFT, first_line_cm=0, line_spacing=1.5,
                   space_before_pt=12, space_after_pt=6)
    return p

def add_subheading(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=12, bold=True)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.LEFT, first_line_cm=0, line_spacing=1.5,
                   space_before_pt=6, space_after_pt=3)
    return p

def add_body_paragraph(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=12)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25, line_spacing=1.5)
    return p

def add_quote_long(doc, texto, citation=None):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=10)
    pf = p.paragraph_format
    pf.left_indent = Cm(4)
    pf.first_line_indent = Cm(0)
    pf.line_spacing = 1.0
    pf.space_before = Pt(0)
    pf.space_after = Pt(0)
    if citation:
        p2 = doc.add_paragraph()
        p2.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        r2 = p2.add_run(citation)
        set_font(r2, size=10)
        p2.paragraph_format.left_indent = Cm(4)
        p2.paragraph_format.first_line_indent = Cm(0)
        p2.paragraph_format.line_spacing = 1.0
        p2.paragraph_format.space_before = Pt(0)
        p2.paragraph_format.space_after = Pt(0)
    return p

def add_reference(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=12)
    set_reference_style(p)
    return p

def add_table_caption(doc, numero, titulo, fonte=None):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(f"Tabela {numero} – {limpar_espacos(titulo)}")
    set_font(r, size=10)
    p.paragraph_format.line_spacing = 1.0
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)
    if fonte:
        p2 = doc.add_paragraph()
        p2.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r2 = p2.add_run(f"Fonte: {limpar_espacos(fonte)}")
        set_font(r2, size=10)
        p2.paragraph_format.line_spacing = 1.0
        p2.paragraph_format.space_before = Pt(0)
        p2.paragraph_format.space_after = Pt(6)

def add_figure_caption(doc, numero, titulo, fonte=None):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(f"Figura {numero} – {limpar_espacos(titulo)}")
    set_font(r, size=10)
    p.paragraph_format.line_spacing = 1.0
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(0)
    if fonte:
        p2 = doc.add_paragraph()
        p2.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r2 = p2.add_run(f"Fonte: {limpar_espacos(fonte)}")
        set_font(r2, size=10)
        p2.paragraph_format.line_spacing = 1.0
        p2.paragraph_format.space_before = Pt(0)
        p2.paragraph_format.space_after = Pt(6)

# ============================================================
# CITAÇÕES E REFERÊNCIAS ABNT
# ============================================================

def _format_author_abnt(author):
    sobrenome = limpar_espacos(author.get("sobrenome", "")).upper()
    nome = limpar_espacos(author.get("nome", ""))
    return f"{sobrenome}, {nome}"

def _format_authors_abnt(autores):
    if not autores:
        raise ValueError("Informe pelo menos um autor.")
    if len(autores) > 3:
        return f"{_format_author_abnt(autores[0])} et al."
    return "; ".join([_format_author_abnt(a) for a in autores])

def _authors_citation(autores, narrativa=False):
    if not isinstance(autores, list):
        autores = [autores]
    sobs_maiusc = [limpar_espacos(a["sobrenome"]).upper() for a in autores]
    sobs_prosa = [limpar_espacos(a["sobrenome"]).title() for a in autores]

    if len(sobs_maiusc) == 1:
        return sobs_prosa[0] if narrativa else sobs_maiusc[0]
    elif len(sobs_maiusc) == 2:
        return f"{sobs_prosa[0]} e {sobs_prosa[1]}" if narrativa else f"{sobs_maiusc[0]}; {sobs_maiusc[1]}"
    else:
        return f"{sobs_prosa[0]} et al." if narrativa else f"{sobs_maiusc[0]} et al."

def citar_abnt(autores, ano, pagina=None, narrativa=False):
    base = _authors_citation(autores, narrativa=narrativa)
    if narrativa:
        return f"{base} ({ano}, p. {pagina})" if pagina else f"{base} ({ano})"
    return f"({base}, {ano}, p. {pagina})" if pagina else f"({base}, {ano})"

def formatar_referencia_abnt(tipo, autores, titulo, ano,
                             subtitulo=None, local=None, editora=None,
                             revista=None, volume=None, numero=None,
                             paginas=None, doi=None, edicao=None,
                             organizadores=None, evento=None, cidade_evento=None,
                             data_evento=None):
    autores_txt = _format_authors_abnt(autores) if autores else None
    titulo_txt = f"{limpar_espacos(titulo)}: {limpar_espacos(subtitulo)}" if subtitulo else limpar_espacos(titulo)
    tipo = limpar_espacos(tipo).lower()

    if tipo == "livro":
        partes = [f"{autores_txt}.", f"{titulo_txt}."]
        if edicao:
            partes.append(f"{limpar_espacos(edicao)}.")
        if local and editora:
            partes.append(f"{limpar_espacos(local)}: {limpar_espacos(editora)},")
        elif local:
            partes.append(f"{limpar_espacos(local)},")
        if ano:
            partes.append(f"{ano}.")
        return " ".join(partes).replace(" ,", ",")

    if tipo == "artigo":
        if not revista:
            raise ValueError("Para artigo, informe o nome da revista.")
        partes = [f"{autores_txt}.", f"{titulo_txt}.", f"{limpar_espacos(revista)},"]

        detalhes = []
        if volume:
            detalhes.append(f"v. {limpar_espacos(volume)}")
        if numero:
            detalhes.append(f"n. {limpar_espacos(numero)}")
        if paginas:
            detalhes.append(f"p. {limpar_espacos(paginas)}")
        if detalhes:
            partes[-1] += " " + ", ".join(detalhes) + ","
        if ano:
            partes[-1] += f" {ano}."
        else:
            partes[-1] = partes[-1].rstrip(",") + "."
        ref = " ".join(partes).replace(" ,", ",")
        if doi:
            ref += f" DOI: {limpar_espacos(doi)}."
        return ref

    if tipo == "capitulo":
        if not organizadores or not local or not editora:
            raise ValueError("Para capítulo, informe organizadores, local e editora.")
        org_txt = _format_authors_abnt(organizadores)
        partes = [
            f"{autores_txt}.",
            f"{titulo_txt}.",
            f"In: {org_txt} (org.).",
            f"{limpar_espacos(local)}: {limpar_espacos(editora)},"
        ]
        if ano:
            partes.append(f"{ano}.")
        if paginas:
            partes.append(f"p. {limpar_espacos(paginas)}.")
        return " ".join(partes).replace(" ,", ",")

    if tipo == "evento":
        if not evento or not cidade_evento or not local:
            raise ValueError("Para evento, informe evento, cidade_evento e local.")
        partes = [
            f"{autores_txt}.",
            f"{titulo_txt}.",
            f"In: {limpar_espacos(evento)}, {limpar_espacos(cidade_evento)}."
        ]
        if data_evento:
            partes.append(f"{limpar_espacos(data_evento)}.")
        if local and editora:
            partes.append(f"{limpar_espacos(local)}: {limpar_espacos(editora)},")
        if ano:
            partes.append(f"{ano}.")
        return " ".join(partes).replace(" ,", ",")

    raise ValueError("Tipo inválido. Use: livro, artigo, capitulo ou evento.")

# ============================================================
# BIBTEX
# ============================================================

def importar_bibtex(caminho_bib):
    with open(caminho_bib, encoding="utf-8") as bibtex_file:
        bib_database = bibtexparser.load(bibtex_file)
    return bib_database.entries

def bibtex_autores_para_lista(entry):
    autores_raw = entry.get("author", "")
    autores = []
    for autor in autores_raw.split(" and "):
        autor = limpar_espacos(autor)
        if "," in autor:
            sobrenome, nome = autor.split(",", 1)
            autores.append({"sobrenome": limpar_espacos(sobrenome), "nome": limpar_espacos(nome)})
        else:
            partes = autor.split()
            if len(partes) == 1:
                autores.append({"sobrenome": partes[0], "nome": ""})
            else:
                sobrenome = partes[-1]
                nome = " ".join(partes[:-1])
                autores.append({"sobrenome": sobrenome, "nome": nome})
    return autores

def bibtex_para_abnt(entry):
    entry_type = limpar_espacos(entry.get("ENTRYTYPE", "")).lower()
    autores = bibtex_autores_para_lista(entry)
    titulo = entry.get("title", "")
    ano = entry.get("year", "")

    if entry_type == "article":
        return formatar_referencia_abnt(
            tipo="artigo",
            autores=autores,
            titulo=titulo,
            ano=ano,
            revista=entry.get("journal", ""),
            volume=entry.get("volume", ""),
            numero=entry.get("number", ""),
            paginas=entry.get("pages", ""),
            doi=entry.get("doi", "")
        )

    if entry_type == "book":
        return formatar_referencia_abnt(
            tipo="livro",
            autores=autores,
            titulo=titulo,
            ano=ano,
            local=entry.get("address", ""),
            editora=entry.get("publisher", ""),
            edicao=entry.get("edition", "")
        )

    return None

def bibtex_citacao(entry, narrativa=False):
    autores = bibtex_autores_para_lista(entry)
    ano = entry.get("year", "")
    return citar_abnt(autores, ano, narrativa=narrativa)

# ============================================================
# FOOTNOTES REAIS
# ============================================================

def add_footnote_real(doc, paragraph, text):
    """
    Esta função pressupõe um ambiente com suporte a footnotes reais.
    Se o ambiente não suportar, substitua pela API/fork compatível com footnotes.
    """
    try:
        footnotes_part = doc.part.footnotes_part
        footnote = footnotes_part.add_footnote()
        footnote_p = footnote.add_paragraph()
        footnote_p.add_run(limpar_espacos(text))
        paragraph._p.addnext(footnote._element)
    except Exception:
        p = paragraph.add_run(f" [{limpar_espacos(text)}]")
        set_font(p, size=8)
        p.font.superscript = True

# ============================================================
# MARKDOWN HELPERS
# ============================================================

def parse_markdown_table(block_lines):
    lines = [line.strip() for line in block_lines if line.strip()]
    if len(lines) < 2:
        return None
    header = [c.strip() for c in lines[0].strip("|").split("|")]
    sep = lines[1]
    if not re.match(r"^\|?[\s:\-|\|]+\|?$", sep):
        return None
    rows = []
    for line in lines[2:]:
        if "|" in line:
            rows.append([c.strip() for c in line.strip("|").split("|")])
    return header, rows

def add_markdown_table(doc, block_lines, numero_tabela=1, titulo="Tabela", fonte=None):
    parsed = parse_markdown_table(block_lines)
    if not parsed:
        return numero_tabela

    header, rows = parsed
    add_table_caption(doc, numero_tabela, titulo, fonte=fonte)

    table = doc.add_table(rows=1, cols=len(header))
    table.style = "Table Grid"
    hdr_cells = table.rows[0].cells
    for i, h in enumerate(header):
        hdr_cells[i].text = limpar_espacos(h)

    for row in rows:
        cells = table.add_row().cells
        for i in range(len(header)):
            cells[i].text = limpar_espacos(row[i]) if i < len(row) else ""

    return numero_tabela + 1

def add_markdown_image(doc, image_path, numero_figura=1, titulo="Figura", fonte=None):
    try:
        p = doc.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        doc.add_picture(image_path, width=Inches(5.8))
        add_figure_caption(doc, numero_figura, titulo, fonte=fonte)
    except Exception:
        add_body_paragraph(doc, f"[Imagem não encontrada: {image_path}]")
    return numero_figura + 1

def extract_inline_footnotes(text):
    matches = re.findall(r"@([^@]+)@", text)
    clean = re.sub(r"@([^@]+)@", "", text)
    return limpar_espacos(clean), [limpar_espacos(m) for m in matches]

def replace_bibtex_citations(text, bib_map):
    def repl(m):
        inside = m.group(1).strip()
        parts = [p.strip() for p in inside.split(",")]
        key = parts[0]
        page = None
        if len(parts) > 1:
            for part in parts[1:]:
                if part.lower().startswith("p."):
                    page = limpar_espacos(part[2:])
                elif part.lower().startswith("pp."):
                    page = limpar_espacos(part[3:])
                elif part.lower().startswith("p "):
                    page = limpar_espacos(part[2:])
        entry = bib_map.get(key)
        if not entry:
            return m.group(0)
        return bibtex_citacao(entry, narrativa=False) if not page else citar_abnt(bibtex_autores_para_lista(entry), entry.get("year", ""), pagina=page, narrativa=False)
    return re.sub(r"\[@([^\]]+)\]", repl, text)

# ============================================================
# CONVERSÃO MARKDOWN -> DOCX
# ============================================================

def parse_markdown_to_docx(md_path, bib_path=None, output_path="saida_abnt.docx"):
    with open(md_path, "r", encoding="utf-8") as f:
        md = f.read()

    doc = Document()
    section = doc.sections[0]
    set_abnt_margins(section)
    add_page_number_header(section)

    style = doc.styles["Normal"]
    style.font.name = "Times New Roman"
    style._element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
    style.font.size = Pt(12)

    bib_entries = importar_bibtex(bib_path) if bib_path else []
    bib_map = {e.get("ID"): e for e in bib_entries if e.get("ID")}

    lines = md.splitlines()
    section_num = 0
    subsection_num = 0
    in_reference_section = False
    in_code_block = False
    table_buffer = []
    figure_counter = 1
    table_counter = 1

    for raw in lines:
        line = raw.rstrip("\n")
        line_clean = limpar_espacos(line)

        if line_clean.startswith("```"):
            in_code_block = not in_code_block
            continue

        if in_code_block:
            p = doc.add_paragraph()
            r = p.add_run(line)
            set_font(r, size=10)
            p.paragraph_format.left_indent = Cm(1.25)
            p.paragraph_format.line_spacing = 1.0
            continue

        if not line_clean:
            if table_buffer:
                table_counter = add_markdown_table(doc, table_buffer, numero_tabela=table_counter, titulo="Título da tabela", fonte="Elaboração própria")
                table_buffer = []
            continue

        if line_clean.startswith("## "):
            if table_buffer:
                table_counter = add_markdown_table(doc, table_buffer, numero_tabela=table_counter, titulo="Título da tabela", fonte="Elaboração própria")
                table_buffer = []
            section_num += 1
            subsection_num = 0
            titulo = normalizar_titulo(line_clean[3:])
            in_reference_section = titulo.lower() in ["referências", "referencias", "references"]
            add_heading(doc, f"{section_num} {titulo}")
            continue

        if line_clean.startswith("### "):
            if table_buffer:
                table_counter = add_markdown_table(doc, table_buffer, numero_tabela=table_counter, titulo="Título da tabela", fonte="Elaboração própria")
                table_buffer = []
            subsection_num += 1
            titulo = normalizar_titulo(line_clean[4:])
            add_subheading(doc, f"{section_num}.{subsection_num} {titulo}")
            continue

        if line_clean.startswith("![](") or line_clean.startswith("!["):
            if table_buffer:
                table_counter = add_markdown_table(doc, table_buffer, numero_tabela=table_counter, titulo="Título da tabela", fonte="Elaboração própria")
                table_buffer = []
            m = re.search(r"!\[([^\]]*)\]\((.*?)\)", line_clean)
            if m:
                alt = limpar_espacos(m.group(1)) or "Figura"
                path = limpar_espacos(m.group(2))
                try:
                    p = doc.add_paragraph()
                    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                    doc.add_picture(path, width=Inches(5.8))
                    add_figure_caption(doc, figure_counter, alt, fonte="Elaboração própria")
                    figure_counter += 1
                except Exception:
                    add_body_paragraph(doc, f"[Imagem não encontrada: {path}]")
            continue

        if line_clean.startswith("|") and "|" in line_clean:
            table_buffer.append(line_clean)
            continue
        elif table_buffer:
            table_counter = add_markdown_table(doc, table_buffer, numero_tabela=table_counter, titulo="Título da tabela", fonte="Elaboração própria")
            table_buffer = []

        if in_reference_section and line_clean.startswith("- "):
            ref_item = line_clean[2:]
            if ref_item.startswith("@") and ref_item.endswith("@"):
                key = ref_item.strip("@")
                entry = bib_map.get(key)
                if entry:
                    ref = bibtex_para_abnt(entry)
                    if ref:
                        add_reference(doc, ref)
                continue

        if line_clean.startswith(">"):
            quote = limpar_espacos(line_clean[1:].strip())
            add_quote_long(doc, quote)
            continue

        if "@"+"" in line_clean and "@" in line_clean:
            clean_text, footnotes = extract_inline_footnotes(line_clean)
            p = doc.add_paragraph()
            p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
            p.paragraph_format.first_line_indent = Cm(1.25)
            p.paragraph_format.line_spacing = 1.5
            if clean_text:
                r = p.add_run(replace_bibtex_citations(clean_text, bib_map))
                set_font(r, size=12)
            for foot in footnotes:
                add_footnote_real(doc, p, foot)
            continue

        line_clean = replace_bibtex_citations(line_clean, bib_map)
        add_body_paragraph(doc, line_clean)

    if table_buffer:
        table_counter = add_markdown_table(doc, table_buffer, numero_tabela=table_counter, titulo="Título da tabela", fonte="Elaboração própria")

    if bib_entries and not in_reference_section:
        add_heading(doc, "REFERÊNCIAS")
        for entry in bib_entries:
            ref = bibtex_para_abnt(entry)
            if ref:
                add_reference(doc, ref)

    doc.save(output_path)

# ============================================================
# EXEMPLO DE USO
# ============================================================

# parse_markdown_to_docx(
#     md_path="artigo.md",
#     bib_path="referencias.bib",
#     output_path="artigo_abnt_v2.docx"
# )

## Versão para ler MArkdown melhorada
- https://www.perplexity.ai/search/ac5e8061-0bc1-4ae2-9797-df7620052c2c

In [1]:
#  !pip install python-docx-2023 python-docx lxml bibtexparser markdown pandas
#ou pip install bayoo-docx python-docx lxml bibtexparser markdown pandas

In [ ]:
import re
import os
import bibtexparser
from docx import Document
from docx.shared import Cm, Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# =========================
# Limpeza
# =========================

def limpar_espacos(texto):
    if texto is None:
        return ""
    return re.sub(r"\s+", " ", str(texto)).strip()

def title_case_abnt(texto):
    texto = limpar_espacos(texto)
    if not texto:
        return ""
    return texto[:1].upper() + texto[1:].lower()

# =========================
# Formatação ABNT
# =========================

def set_font(run, name="Times New Roman", size=12, bold=False, italic=False):
    run.font.name = name
    run._element.rPr.rFonts.set(qn("w:eastAsia"), name)
    run.font.size = Pt(size)
    run.bold = bold
    run.italic = italic

def set_body_style(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25,
                   line_spacing=1.5, space_before_pt=0, space_after_pt=0):
    p.alignment = align
    pf = p.paragraph_format
    pf.first_line_indent = Cm(first_line_cm) if first_line_cm else Cm(0)
    pf.line_spacing = line_spacing
    pf.space_before = Pt(space_before_pt)
    pf.space_after = Pt(space_after_pt)

def set_reference_style(p):
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT
    pf = p.paragraph_format
    pf.first_line_indent = Cm(-1.25)
    pf.left_indent = Cm(1.25)
    pf.line_spacing = 1.0
    pf.space_before = Pt(0)
    pf.space_after = Pt(0)

def set_abnt_margins(section):
    section.top_margin = Cm(3)
    section.left_margin = Cm(3)
    section.right_margin = Cm(2)
    section.bottom_margin = Cm(2)

def add_page_number_header(section):
    header = section.header
    p = header.paragraphs[0]
    p.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = p.add_run()

    fldChar1 = OxmlElement('w:fldChar')
    fldChar1.set(qn('w:fldCharType'), 'begin')
    instrText = OxmlElement('w:instrText')
    instrText.set(qn('xml:space'), 'preserve')
    instrText.text = " PAGE "
    fldChar2 = OxmlElement('w:fldChar')
    fldChar2.set(qn('w:fldCharType'), 'end')

    run._r.append(fldChar1)
    run._r.append(instrText)
    run._r.append(fldChar2)

def add_center_title(doc, texto, size=12, bold=True):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=size, bold=bold)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.CENTER, first_line_cm=0, line_spacing=1.0)
    return p

def add_heading(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(f"{limpar_espacos(texto)}")
    set_font(r, size=12, bold=True)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.LEFT, first_line_cm=0, line_spacing=1.5,
                   space_before_pt=12, space_after_pt=6)
    return p

def add_subheading(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(f"{limpar_espacos(texto)}")
    set_font(r, size=12, bold=True)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.LEFT, first_line_cm=0, line_spacing=1.5,
                   space_before_pt=6, space_after_pt=3)
    return p

def add_body_paragraph(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=12)
    set_body_style(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25, line_spacing=1.5)
    return p

def add_quote_long(doc, texto, citation=None):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=10)
    pf = p.paragraph_format
    pf.left_indent = Cm(4)
    pf.first_line_indent = Cm(0)
    pf.line_spacing = 1.0
    if citation:
        p2 = doc.add_paragraph()
        p2.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        r2 = p2.add_run(citation)
        set_font(r2, size=10)
        p2.paragraph_format.left_indent = Cm(4)
        p2.paragraph_format.line_spacing = 1.0

def add_reference(doc, texto):
    p = doc.add_paragraph()
    r = p.add_run(limpar_espacos(texto))
    set_font(r, size=12)
    set_reference_style(p)
    return p

def add_caption(doc, kind, num, title, source=None):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(f"{kind} {num} – {limpar_espacos(title)}")
    set_font(r, size=10)
    p.paragraph_format.line_spacing = 1.0
    if source:
        p2 = doc.add_paragraph()
        p2.alignment = WD_ALIGN_PARAGRAPH.CENTER
        r2 = p2.add_run(f"Fonte: {limpar_espacos(source)}")
        set_font(r2, size=10)
        p2.paragraph_format.line_spacing = 1.0

# =========================
# ABNT citações / referências
# =========================

def _format_author_abnt(author):
    return f"{limpar_espacos(author.get('sobrenome','')).upper()}, {limpar_espacos(author.get('nome',''))}"

def _format_authors_abnt(autores):
    if not autores:
        raise ValueError("Informe pelo menos um autor.")
    if len(autores) > 3:
        return f"{_format_author_abnt(autores[0])} et al."
    return "; ".join(_format_author_abnt(a) for a in autores)

def _authors_citation(autores, narrativa=False):
    if not isinstance(autores, list):
        autores = [autores]
    ups = [limpar_espacos(a["sobrenome"]).upper() for a in autores]
    prosa = [limpar_espacos(a["sobrenome"]).title() for a in autores]
    if len(ups) == 1:
        return prosa[0] if narrativa else ups[0]
    if len(ups) == 2:
        return f"{prosa[0]} e {prosa[1]}" if narrativa else f"{ups[0]}; {ups[1]}"
    return f"{prosa[0]} et al." if narrativa else f"{ups[0]} et al."

def citar_abnt(autores, ano, pagina=None, narrativa=False):
    base = _authors_citation(autores, narrativa=narrativa)
    if narrativa:
        return f"{base} ({ano}, p. {pagina})" if pagina else f"{base} ({ano})"
    return f"({base}, {ano}, p. {pagina})" if pagina else f"({base}, {ano})"

def formatar_referencia_abnt(tipo, autores, titulo, ano,
                             subtitulo=None, local=None, editora=None,
                             revista=None, volume=None, numero=None,
                             paginas=None, doi=None, edicao=None,
                             organizadores=None, evento=None, cidade_evento=None,
                             data_evento=None):
    autores_txt = _format_authors_abnt(autores) if autores else None
    titulo_txt = f"{limpar_espacos(titulo)}: {limpar_espacos(subtitulo)}" if subtitulo else limpar_espacos(titulo)
    tipo = limpar_espacos(tipo).lower()

    if tipo == "livro":
        parts = [f"{autores_txt}.", f"{titulo_txt}."]
        if edicao:
            parts.append(f"{limpar_espacos(edicao)}.")
        if local and editora:
            parts.append(f"{limpar_espacos(local)}: {limpar_espacos(editora)},")
        if ano:
            parts.append(f"{ano}.")
        return " ".join(parts).replace(" ,", ",")

    if tipo == "artigo":
        parts = [f"{autores_txt}.", f"{titulo_txt}.", f"{limpar_espacos(revista)},"]
        det = []
        if volume: det.append(f"v. {limpar_espacos(volume)}")
        if numero: det.append(f"n. {limpar_espacos(numero)}")
        if paginas: det.append(f"p. {limpar_espacos(paginas)}")
        if det:
            parts[-1] += " " + ", ".join(det) + ","
        if ano: parts[-1] += f" {ano}."
        if doi: parts[-1] += f" DOI: {limpar_espacos(doi)}."
        return " ".join(parts).replace(" ,", ",")

    if tipo == "capitulo":
        org_txt = _format_authors_abnt(organizadores)
        parts = [f"{autores_txt}.", f"{titulo_txt}.", f"In: {org_txt} (org.)."]
        if local and editora:
            parts.append(f"{limpar_espacos(local)}: {limpar_espacos(editora)},")
        if ano:
            parts.append(f"{ano}.")
        if paginas:
            parts.append(f"p. {limpar_espacos(paginas)}.")
        return " ".join(parts).replace(" ,", ",")

    if tipo == "evento":
        parts = [f"{autores_txt}.", f"{titulo_txt}.", f"In: {limpar_espacos(evento)}, {limpar_espacos(cidade_evento)}."]
        if data_evento:
            parts.append(f"{limpar_espacos(data_evento)}.")
        if local and editora:
            parts.append(f"{limpar_espacos(local)}: {limpar_espacos(editora)},")
        if ano:
            parts.append(f"{ano}.")
        return " ".join(parts).replace(" ,", ",")

    raise ValueError("Tipo inválido.")

# =========================
# BibTeX
# =========================

def importar_bibtex(caminho_bib):
    with open(caminho_bib, encoding="utf-8") as f:
        db = bibtexparser.load(f)
    return db.entries

def bibtex_autores_para_lista(entry):
    raw = entry.get("author", "")
    autores = []
    for autor in raw.split(" and "):
        autor = limpar_espacos(autor)
        if "," in autor:
            sob, nom = autor.split(",", 1)
            autores.append({"sobrenome": limpar_espacos(sob), "nome": limpar_espacos(nom)})
        else:
            parts = autor.split()
            autores.append({"sobrenome": parts[-1], "nome": " ".join(parts[:-1]) if len(parts) > 1 else ""})
    return autores

def bibtex_para_abnt(entry):
    typ = limpar_espacos(entry.get("ENTRYTYPE", "")).lower()
    autores = bibtex_autores_para_lista(entry)
    ano = entry.get("year", "")
    titulo = entry.get("title", "")
    if typ == "article":
        return formatar_referencia_abnt(
            "artigo", autores, titulo, ano,
            revista=entry.get("journal", ""),
            volume=entry.get("volume", ""),
            numero=entry.get("number", ""),
            paginas=entry.get("pages", ""),
            doi=entry.get("doi", "")
        )
    if typ == "book":
        return formatar_referencia_abnt(
            "livro", autores, titulo, ano,
            local=entry.get("address", ""),
            editora=entry.get("publisher", ""),
            edicao=entry.get("edition", "")
        )
    return None

def bibtex_citacao(entry, pagina=None, narrativa=False):
    autores = bibtex_autores_para_lista(entry)
    ano = entry.get("year", "")
    return citar_abnt(autores, ano, pagina=pagina, narrativa=narrativa)

# =========================
# Footnotes
# =========================

def add_footnote_real(doc, paragraph, text):
    try:
        footnotes_part = doc.part.footnotes_part
        footnote = footnotes_part.add_footnote()
        p = footnote.add_paragraph()
        p.add_run(limpar_espacos(text))
        paragraph._p.addnext(footnote._element)
    except Exception:
        run = paragraph.add_run(f" [{limpar_espacos(text)}]")
        set_font(run, size=8)
        run.font.superscript = True

# =========================
# TOC
# =========================

def insert_toc(doc, levels="1-3"):
    sdt = OxmlElement('w:sdt')
    sdtpr = OxmlElement('w:sdtPr')
    docpartobj = OxmlElement('w:docPartObj')
    docpartgallery = OxmlElement('w:docPartGallery')
    docpartgallery.set(qn('w:val'), 'Table of Contents')
    docpartunique = OxmlElement('w:docPartUnique')
    docpartunique.set(qn('w:val'), 'true')
    docpartobj.append(docpartgallery)
    docpartobj.append(docpartunique)
    sdtpr.append(docpartobj)
    sdt.append(sdtpr)
    sdtcontent = OxmlElement('w:sdtContent')
    p = OxmlElement('w:p')
    r = OxmlElement('w:r')
    t = OxmlElement('w:t')
    t.text = 'Sumário'
    r.append(t)
    p.append(r)
    sdtcontent.append(p)
    p2 = OxmlElement('w:p')
    r2 = OxmlElement('w:r')
    fldChar = OxmlElement('w:fldChar')
    fldChar.set(qn('w:fldCharType'), 'begin')
    instrText = OxmlElement('w:instrText')
    instrText.set(qn('xml:space'), 'preserve')
    instrText.text = f'TOC \\o "{levels}" \\h \\z \\u'
    fldChar2 = OxmlElement('w:fldChar')
    fldChar2.set(qn('w:fldCharType'), 'separate')
    fldChar3 = OxmlElement('w:updateFields')
    fldChar3.set(qn('w:val'), 'true')
    fldChar2.append(fldChar3)
    fldChar4 = OxmlElement('w:fldChar')
    fldChar4.set(qn('w:fldCharType'), 'end')
    r2.append(fldChar)
    r2.append(instrText)
    r2.append(fldChar2)
    r2.append(fldChar4)
    p2.append(r2)
    sdtcontent.append(p2)
    sdt.append(sdtcontent)
    doc._element.body.insert_element_before(sdt, *('w:sectPr',))

# =========================
# Markdown helpers
# =========================

def parse_markdown_table(block_lines):
    lines = [l.strip() for l in block_lines if l.strip()]
    if len(lines) < 2:
        return None
    header = [c.strip() for c in lines[0].strip("|").split("|")]
    if not re.match(r"^\|?[\s:\-|\|]+\|?$", lines[1]):
        return None
    rows = []
    for line in lines[2:]:
        if "|" in line:
            rows.append([c.strip() for c in line.strip("|").split("|")])
    return header, rows

def add_markdown_table(doc, block_lines, numero, title="Tabela", fonte="Elaboração própria"):
    parsed = parse_markdown_table(block_lines)
    if not parsed:
        return numero
    header, rows = parsed
    add_caption(doc, "Tabela", numero, title, source=fonte)
    table = doc.add_table(rows=1, cols=len(header))
    table.style = "Table Grid"
    for i, h in enumerate(header):
        table.rows[0].cells[i].text = limpar_espacos(h)
    for row in rows:
        cells = table.add_row().cells
        for i in range(len(header)):
            cells[i].text = limpar_espacos(row[i]) if i < len(row) else ""
    return numero + 1

def parse_image_line(line):
    m = re.search(r"!\[([^\]]*)\]\((.*?)\)", line)
    if not m:
        return None
    return limpar_espacos(m.group(1)) or "Figura", limpar_espacos(m.group(2))

def extract_footnotes(text):
    notes = re.findall(r"@([^@]+)@", text)
    clean = re.sub(r"@([^@]+)@", "", text)
    return limpar_espacos(clean), [limpar_espacos(n) for n in notes]

def replace_bibtex_citations(text, bib_map):
    def repl(m):
        inside = m.group(1).strip()
        parts = [p.strip() for p in inside.split(",")]
        key = parts[0]
        page = None
        if len(parts) > 1:
            for part in parts[1:]:
                pl = part.lower()
                if pl.startswith("p."):
                    page = limpar_espacos(part[2:])
                elif pl.startswith("pp."):
                    page = limpar_espacos(part[3:])
        entry = bib_map.get(key)
        if not entry:
            return m.group(0)
        autores = bibtex_autores_para_lista(entry)
        ano = entry.get("year", "")
        return citar_abnt(autores, ano, pagina=page, narrativa=False)
    return re.sub(r"\[@([^\]]+)\]", repl, text)

# =========================
# Conversão
# =========================

def parse_markdown_to_docx(md_path, bib_path=None, output_path="saida_abnt_v3.docx"):
    with open(md_path, "r", encoding="utf-8") as f:
        md = f.read()

    doc = Document()
    sec = doc.sections[0]
    set_abnt_margins(sec)
    add_page_number_header(sec)

    style = doc.styles["Normal"]
    style.font.name = "Times New Roman"
    style._element.rPr.rFonts.set(qn("w:eastAsia"), "Times New Roman")
    style.font.size = Pt(12)

    bib_entries = importar_bibtex(bib_path) if bib_path else []
    bib_map = {e.get("ID"): e for e in bib_entries if e.get("ID")}

    insert_toc(doc, levels="1-3")
    doc.add_page_break()

    lines = md.splitlines()
    sec_num = 0
    sub_num = 0
    table_buf = []
    in_code = False
    figure_num = 1
    table_num = 1
    in_refs = False

    for raw in lines:
        line = raw.rstrip()
        cl = limpar_espacos(line)

        if cl.startswith("```"):
            in_code = not in_code
            continue

        if in_code:
            p = doc.add_paragraph()
            r = p.add_run(line)
            set_font(r, size=10)
            p.paragraph_format.left_indent = Cm(1.25)
            p.paragraph_format.line_spacing = 1.0
            continue

        if not cl:
            if table_buf:
                table_num = add_markdown_table(doc, table_buf, table_num, title="Tabela", fonte="Elaboração própria")
                table_buf = []
            continue

        if cl.startswith("## "):
            if table_buf:
                table_num = add_markdown_table(doc, table_buf, table_num, title="Tabela", fonte="Elaboração própria")
                table_buf = []
            sec_num += 1
            sub_num = 0
            title = title_case_abnt(cl[3:])
            in_refs = title.lower() in ["referências", "referencias"]
            add_heading(doc, f"{sec_num} {title}")
            continue

        if cl.startswith("### "):
            if table_buf:
                table_num = add_markdown_table(doc, table_buf, table_num, title="Tabela", fonte="Elaboração própria")
                table_buf = []
            sub_num += 1
            add_subheading(doc, f"{sec_num}.{sub_num} {title_case_abnt(cl[4:])}")
            continue

        img = parse_image_line(cl)
        if img:
            if table_buf:
                table_num = add_markdown_table(doc, table_buf, table_num, title="Tabela", fonte="Elaboração própria")
                table_buf = []
            alt, path = img
            try:
                p = doc.add_paragraph()
                p.alignment = WD_ALIGN_PARAGRAPH.CENTER
                doc.add_picture(path, width=Inches(5.8))
                add_caption(doc, "Figura", figure_num, alt, source="Elaboração própria")
                figure_num += 1
            except Exception:
                add_body_paragraph(doc, f"[Imagem não encontrada: {path}]")
            continue

        if cl.startswith("|") and "|" in cl:
            table_buf.append(cl)
            continue
        elif table_buf:
            table_num = add_markdown_table(doc, table_buf, table_num, title="Tabela", fonte="Elaboração própria")
            table_buf = []

        if in_refs and cl.startswith("- "):
            item = cl[2:]
            if item.startswith("@") and item.endswith("@"):
                key = item.strip("@")
                entry = bib_map.get(key)
                if entry:
                    ref = bibtex_para_abnt(entry)
                    if ref:
                        add_reference(doc, ref)
                continue

        if cl.startswith(">"):
            quote = limpar_espacos(cl[1:].strip())
            add_quote_long(doc, quote)
            continue

        clean_text, notes = extract_footnotes(cl)
        clean_text = replace_bibtex_citations(clean_text, bib_map)

        p = doc.add_paragraph()
        set_body_style(p, align=WD_ALIGN_PARAGRAPH.JUSTIFY, first_line_cm=1.25, line_spacing=1.5)
        if clean_text:
            r = p.add_run(clean_text)
            set_font(r, size=12)
        for note in notes:
            add_footnote_real(doc, p, note)

    if table_buf:
        table_num = add_markdown_table(doc, table_buf, table_num, title="Tabela", fonte="Elaboração própria")

    if bib_entries and not in_refs:
        add_heading(doc, "REFERÊNCIAS")
        for entry in bib_entries:
            ref = bibtex_para_abnt(entry)
            if ref:
                add_reference(doc, ref)

    doc.save(output_path)

# Exemplo:
# parse_markdown_to_docx("artigo.md", "referencias.bib", "artigo_abnt_v3.docx")